# Divvy Bikeshare Lakehouse — 01 · Bronze (Extract + Load)

**Author:** Tarie Nosworthy

**Run order:** `01_bronze_extract_load` → `02_gold_dimensions` → `03_gold_facts`

| Step | What happens here |
|---|---|
| **Extract** | Read the four Divvy CSV files out of the Databricks file system and write them to **Delta files** under `/delta/bronze/...` |
| **Load** | Use `spark.sql` to create the **bronze database and tables** on top of those Delta files |

The bronze store is a faithful, typed copy of the source system — no business logic,
no joins, no filtering, nothing dropped. All modelling happens downstream in the gold
notebooks, so bronze stays re-runnable and auditable against the raw CSVs.

## 0 · Configuration

Upload the four project CSVs to DBFS first — `Data` → `Create Table` → `Upload File`,
or the DBFS file browser (enable it under `Admin Console` → `Workspace Settings` →
`Advanced` → `DBFS File Browser`). Uploads land in `dbfs:/FileStore/tables/` by default.

`trips.csv` is ~440 MB, which is a slow browser upload. Spark reads gzip directly, so
uploading `trips.csv.gz` instead is roughly a quarter of the bytes and needs no code
change — the file resolver below accepts either.

In [0]:
SOURCE_DIR = "dbfs:/FileStore/tables"      # where the raw CSVs were uploaded
BRONZE_DIR = "dbfs:/delta/bronze"          # where the bronze Delta files are written
BRONZE_DB = "bronze"                       # bronze database (the bronze data store)

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DateType, TimestampType, BooleanType, DecimalType, DoubleType,
)

spark.conf.set("spark.sql.shuffle.partitions", 8)   # small data, single-node cluster

# Unity Catalog refuses to create tables over `dbfs:` locations
# (UC_FILE_SCHEME_FOR_TABLE_CREATION_NOT_SUPPORTED). Both data stores live in DBFS, so
# pin the session to the workspace's Hive metastore, where DBFS-backed external tables
# are supported. Workspaces without Unity Catalog have no such catalog and need no pin.
try:
    spark.sql("USE CATALOG hive_metastore")
    print("catalog: hive_metastore")
except Exception:
    print("catalog: workspace default (no Unity Catalog here)")

catalog: hive_metastore


## 1 · Locate the source files

File names in the project archive have varied (`riders.csv` vs `rider.csv`), so each is
resolved by keyword rather than hard-coded. This also turns "I forgot to upload one"
into a readable error at the top of the notebook instead of a confusing empty table
three cells later.

In [0]:
def find_csv(directory, keyword):
    """Return the path of the CSV (or .csv.gz) in `directory` whose name contains `keyword`."""
    try:
        entries = dbutils.fs.ls(directory)
    except Exception:
        raise FileNotFoundError(
            "Source directory " + directory + " does not exist. "
            "Upload the project CSVs to DBFS and update SOURCE_DIR."
        )
    matches = [
        f.path for f in entries
        if keyword in f.name.lower()
        and (f.name.lower().endswith(".csv") or f.name.lower().endswith(".csv.gz"))
    ]
    if not matches:
        available = ", ".join(f.name for f in entries) or "(directory is empty)"
        raise FileNotFoundError(
            "No CSV matching '" + keyword + "' in " + directory + ". Found: " + available
        )
    return matches[0]


SOURCE_FILES = {
    "rider":   find_csv(SOURCE_DIR, "rider"),
    "payment": find_csv(SOURCE_DIR, "payment"),
    "station": find_csv(SOURCE_DIR, "station"),
    "trip":    find_csv(SOURCE_DIR, "trip"),
}

for name, path in SOURCE_FILES.items():
    print(name.ljust(8), "->", path)

rider    -> dbfs:/FileStore/tables/riders.csv
payment  -> dbfs:/FileStore/tables/payments.csv
station  -> dbfs:/FileStore/tables/stations.csv
trip     -> dbfs:/FileStore/tables/trips.csv


## 2 · Source schemas

The files are headerless, so column order comes from the relational ERD supplied with
the project. Declaring the schema explicitly also skips the inference pass, which on a
440 MB `trips.csv` is a whole extra read of the file.

One deliberate correction to the ERD: it types `Station.station_id` as `varchar` but
`Trip.start_station_id` / `Trip.end_station_id` as `int`. A key and its foreign key
cannot be different types, and the data settles the argument — `stations.csv` contains
both `525` and `KA1503000012`. **Station identifiers are strings throughout.**

In [0]:
RIDER_SCHEMA = StructType([
    StructField("rider_id",           IntegerType()),
    StructField("first",              StringType()),
    StructField("last",               StringType()),
    StructField("address",            StringType()),
    StructField("birthday",           DateType()),
    StructField("account_start_date", DateType()),
    StructField("account_end_date",   DateType()),
    StructField("is_member",          BooleanType()),
])

PAYMENT_SCHEMA = StructType([
    StructField("payment_id", IntegerType()),
    StructField("date",       DateType()),
    StructField("amount",     DecimalType(10, 2)),
    StructField("rider_id",   IntegerType()),
])

STATION_SCHEMA = StructType([
    StructField("station_id", StringType()),
    StructField("name",       StringType()),
    StructField("latitude",   DoubleType()),
    StructField("longitude",  DoubleType()),
])

TRIP_SCHEMA = StructType([
    StructField("trip_id",          StringType()),
    StructField("rideable_type",    StringType()),
    StructField("started_at",       TimestampType()),
    StructField("ended_at",         TimestampType()),
    StructField("start_station_id", StringType()),
    StructField("end_station_id",   StringType()),
    StructField("rider_id",         IntegerType()),
])

SCHEMAS = {
    "rider":   RIDER_SCHEMA,
    "payment": PAYMENT_SCHEMA,
    "station": STATION_SCHEMA,
    "trip":    TRIP_SCHEMA,
}

## 3 · Reader

The project files ship **without** a header row, but some copies in circulation have
one. Rather than guess, each file is read as raw strings, any row that turns out to be
a repeat of the column names is dropped, and only then are the declared types applied.
The ingest is correct either way, and a stray header can never masquerade as data.

In [0]:
def read_source_csv(path, schema):
    """Read a headerless (or header-bearing) CSV and return it typed to `schema`."""
    names = [f.name for f in schema.fields]
    string_schema = StructType([StructField(n, StringType()) for n in names])

    raw = (
        spark.read
        .option("header", "false")
        .option("mode", "PERMISSIVE")
        .schema(string_schema)
        .csv(path)
    )

    # Drop a header row if this copy of the file happens to have one.
    first_col = names[0]
    raw = raw.filter(F.lower(F.trim(F.col(first_col))) != F.lit(first_col.lower()))

    # Trim, turn empty strings into NULL, then cast to the declared types.
    return raw.select(*[
        F.when(F.length(F.trim(F.col(f.name))) == 0, None)
         .otherwise(F.trim(F.col(f.name)))
         .cast(f.dataType)
         .alias(f.name)
        for f in schema.fields
    ])

## 4 · EXTRACT — CSV ➜ Delta files

Each source file is written to its own Delta location. This is the extract step the
rubric asks for: data picked up from the Databricks file system and written out to
Delta file locations.

In [0]:
bronze_counts = {}

for name, path in SOURCE_FILES.items():
    df = read_source_csv(path, SCHEMAS[name])
    target = BRONZE_DIR + "/" + name

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .save(target))

    bronze_counts[name] = spark.read.format("delta").load(target).count()
    print("extracted", name.ljust(8), format(bronze_counts[name], ",").rjust(10),
          "rows ->", target)

extracted rider        75,000 rows -> dbfs:/delta/bronze/rider
extracted payment   1,946,607 rows -> dbfs:/delta/bronze/payment
extracted station         838 rows -> dbfs:/delta/bronze/station
extracted trip      4,584,921 rows -> dbfs:/delta/bronze/trip


## 5 · LOAD — Delta files ➜ bronze tables

`spark.sql` creates the bronze database and registers a table over each Delta location
written above, so the raw layer is queryable as SQL. The tables are external: the Delta
files stay exactly where the extract step put them.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS " + BRONZE_DB)

for name in SOURCE_FILES:
    spark.sql("DROP TABLE IF EXISTS " + BRONZE_DB + "." + name)
    spark.sql(
        "CREATE TABLE " + BRONZE_DB + "." + name + " "
        "USING DELTA "
        "LOCATION '" + BRONZE_DIR + "/" + name + "'"
    )
    print("created table", BRONZE_DB + "." + name)

spark.sql("SHOW TABLES IN " + BRONZE_DB).show(truncate=False)

created table bronze.rider
created table bronze.payment
created table bronze.station
created table bronze.trip
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|bronze  |payment  |false      |
|bronze  |rider    |false      |
|bronze  |station  |false      |
|bronze  |trip     |false      |
+--------+---------+-----------+



## 6 · Verify the bronze store

Row counts, then a null check on every column the gold layer joins or keys on. A
problem caught here costs one cell; the same problem caught after the facts are built
costs a full rebuild.

In [0]:
for name in SOURCE_FILES:
    n = spark.table(BRONZE_DB + "." + name).count()
    print((BRONZE_DB + "." + name).ljust(16), format(n, ",").rjust(10), "rows")

bronze.rider         75,000 rows
bronze.payment    1,946,607 rows
bronze.station          838 rows
bronze.trip       4,584,921 rows


In [0]:
spark.sql("""
    SELECT 'rider.rider_id'     AS column_checked, COUNT(*) AS null_rows FROM bronze.rider   WHERE rider_id   IS NULL
    UNION ALL SELECT 'rider.birthday',      COUNT(*) FROM bronze.rider   WHERE birthday   IS NULL
    UNION ALL SELECT 'rider.is_member',     COUNT(*) FROM bronze.rider   WHERE is_member  IS NULL
    UNION ALL SELECT 'payment.payment_id',  COUNT(*) FROM bronze.payment WHERE payment_id IS NULL
    UNION ALL SELECT 'payment.rider_id',    COUNT(*) FROM bronze.payment WHERE rider_id   IS NULL
    UNION ALL SELECT 'payment.date',        COUNT(*) FROM bronze.payment WHERE date       IS NULL
    UNION ALL SELECT 'payment.amount',      COUNT(*) FROM bronze.payment WHERE amount     IS NULL
    UNION ALL SELECT 'station.station_id',  COUNT(*) FROM bronze.station WHERE station_id IS NULL
    UNION ALL SELECT 'trip.trip_id',        COUNT(*) FROM bronze.trip    WHERE trip_id    IS NULL
    UNION ALL SELECT 'trip.started_at',     COUNT(*) FROM bronze.trip    WHERE started_at IS NULL
    UNION ALL SELECT 'trip.ended_at',       COUNT(*) FROM bronze.trip    WHERE ended_at   IS NULL
    UNION ALL SELECT 'trip.rider_id',       COUNT(*) FROM bronze.trip    WHERE rider_id   IS NULL
    UNION ALL SELECT 'trip.start_station_id', COUNT(*) FROM bronze.trip  WHERE start_station_id IS NULL
    UNION ALL SELECT 'trip.end_station_id',   COUNT(*) FROM bronze.trip  WHERE end_station_id   IS NULL
""").show(20, truncate=False)

+---------------------+---------+
|column_checked       |null_rows|
+---------------------+---------+
|rider.rider_id       |0        |
|rider.birthday       |0        |
|rider.is_member      |0        |
|payment.payment_id   |0        |
|payment.rider_id     |0        |
|payment.date         |0        |
|payment.amount       |0        |
|station.station_id   |0        |
|trip.trip_id         |0        |
|trip.started_at      |0        |
|trip.ended_at        |0        |
|trip.rider_id        |0        |
|trip.start_station_id|0        |
|trip.end_station_id  |0        |
+---------------------+---------+



In [0]:
%sql
SELECT * FROM bronze.rider LIMIT 10

rider_id,first,last,address,birthday,account_start_date,account_end_date,is_member
1000,Diana,Clark,1200 Alyssa Squares,1989-02-13,2019-04-23,null,true
1001,Jennifer,Smith,397 Diana Ferry,1976-08-10,2019-11-01,2020-09-01,true
1002,Karen,Smith,644 Brittany Row Apt. 097,1998-08-10,2022-02-04,null,true
1003,Bryan,Roberts,996 Dickerson Turnpike,1999-03-29,2019-08-26,null,false
1004,Jesse,Middleton,7009 Nathan Expressway,1969-04-11,2019-09-14,null,true
1005,Christine,Rodriguez,224 Washington Mills Apt. 467,1974-08-27,2020-03-24,null,false
1006,Alicia,Taylor,1137 Angela Locks,2004-01-30,2020-11-27,2021-12-01,true
1007,Benjamin,Fernandez,979 Phillips Ways,1988-01-11,2016-12-11,null,false
1008,John,Crawford,7691 Evans Court,1987-02-21,2021-03-28,2021-07-01,true
1009,Victoria,Ritter,9922 Jim Crest Apt. 319,1981-02-07,2020-06-12,2021-11-01,true


In [0]:
%sql
SELECT * FROM bronze.payment LIMIT 10

payment_id,date,amount,rider_id
1,2019-05-01,9.00,1000
2,2019-06-01,9.00,1000
3,2019-07-01,9.00,1000
4,2019-08-01,9.00,1000
5,2019-09-01,9.00,1000
6,2019-10-01,9.00,1000
7,2019-11-01,9.00,1000
8,2019-12-01,9.00,1000
9,2020-01-01,9.00,1000
10,2020-02-01,9.00,1000


In [0]:
%sql
SELECT * FROM bronze.station LIMIT 10

station_id,name,latitude,longitude
525,Glenwood Ave & Touhy Ave,42.012701,-87.66605799999999
KA1503000012,Clark St & Lake St,41.88579466666667,-87.63110066666668
637,Wood St & Chicago Ave,41.895634,-87.672069
13216,State St & 33rd St,41.8347335,-87.6258275
18003,Fairbanks St & Superior St,41.89580766666667,-87.62025316666669
KP1705001026,LaSalle Dr & Huron St,41.894877,-87.632326
13253,Lincoln Ave & Waveland Ave,41.948797,-87.675278
KA1503000044,Rush St & Hubbard St,41.890173,-87.62618499999999
KA1504000140,Winchester Ave & Elston Ave,41.92403733333333,-87.67641483333334
TA1305000032,Clinton St & Madison St,41.882242,-87.64106600000001


In [0]:
%sql
SELECT * FROM bronze.trip LIMIT 10

trip_id,rideable_type,started_at,ended_at,start_station_id,end_station_id,rider_id
89E7AA6C29227EFF,classic_bike,2021-02-12T16:14:56Z,2021-02-12T16:21:43Z,525,660,71934
0FEFDE2603568365,classic_bike,2021-02-14T17:52:38Z,2021-02-14T18:12:09Z,525,16806,47854
E6159D746B2DBB91,electric_bike,2021-02-09T19:10:18Z,2021-02-09T19:19:10Z,KA1503000012,TA1305000029,70870
B32D3199F1C2E75B,classic_bike,2021-02-02T17:49:41Z,2021-02-02T17:54:06Z,637,TA1305000034,58974
83E463F23575F4BF,electric_bike,2021-02-23T15:07:23Z,2021-02-23T15:22:37Z,13216,TA1309000055,39608
BDAA7E3494E8D545,electric_bike,2021-02-24T15:43:33Z,2021-02-24T15:49:05Z,18003,KP1705001026,36267
A772742351171257,classic_bike,2021-02-01T17:47:42Z,2021-02-01T17:48:33Z,KP1705001026,KP1705001026,50104
295476889D9B79F8,classic_bike,2021-02-11T18:33:53Z,2021-02-11T18:35:09Z,18003,18003,19618
362087194BA4CC9A,classic_bike,2021-02-27T15:13:39Z,2021-02-27T15:36:36Z,KP1705001026,KP1705001026,16732
21630F715038CCB0,classic_bike,2021-02-20T08:59:42Z,2021-02-20T09:17:04Z,KP1705001026,KP1705001026,57068


### Bronze complete

`bronze.rider`, `bronze.payment`, `bronze.station` and `bronze.trip` are registered as
Delta tables over the files in `/delta/bronze/`. Continue with **`02_gold_dimensions`**.